In [ ]:
!pip install cgodme

In [2]:
import pandas as pd
import numpy as np
from cgodme import data_generation
from cgodme import run_optimization

In [3]:
od_target = pd.read_csv("./user_data/target_demand_auto.csv")
ue_pathflows = pd.read_csv("./user_data/route_assignment.csv")
ue_linkflows = pd.read_csv("./user_data/link.csv")
link_target = pd.read_csv("./user_data/link_performance.csv")
data_source = data_generation(od_target, ue_pathflows, ue_linkflows, link_target)


Imputing the od flow target data to remove unmapped OD pairs...
 - Before removing, the unmapped target od volume is 324540
 - After processing, the target od volume is 300780
Number of Origins: 24
Number of Origin-Destination Pairs: 493
Number of Paths: 1536
Number of Links: 76


In [4]:
data_source=data_generation(od_target, ue_pathflows, ue_linkflows, link_target)


Imputing the od flow target data to remove unmapped OD pairs...
 - Before removing, the unmapped target od volume is 324540
 - After processing, the target od volume is 300780
Number of Origins: 24
Number of Origin-Destination Pairs: 493
Number of Paths: 1536
Number of Links: 76


In [5]:
# get the required data sources (e.g., mapping matrices, initialized od/path flows, bpr function parameters)
data_source = data_generation(od_target, ue_pathflows, ue_linkflows, link_target)

init_od_volume, spare_od_path_inc, path_link_inc, _, = data_source.reformed_incidence_mat()
init_path_flow = data_source.get_init_path_values()
bpr_params = data_source.get_bpr_params()
layer_mapping_matrix = {"o_od_inc": data_source.get_o_to_od_incidence_mat(), 
                        "od_path_inc": spare_od_path_inc}

# define the target data
target_data = {"observed_o_volume": np.array(data_source.o_target_data["volume"], dtype="f"), 
               "observed_od_volume": np.array(data_source.od_target_data["volume"], dtype="f"), 
               "link_count_car": np.array(data_source.link_target_data["link_count_car_volumes"], dtype="f"), 
               "VMT_car": np.array(2700.0, dtype="f")}

# set up the opimization training configuration
optimization_setting = {}
optimization_setting["training_steps"] = 2000
optimization_setting["learning_rates"] = 0.01
optimization_setting["penalty_coefficient"]= 2.0

# set up the multi-objective function scale parameters (0 - 1.0)
obj_setting = {}
obj_setting["passenger_car_count"] = 1.0
obj_setting["passenger_car_vmt"] = 1.0
obj_setting["od_split"] = 1.0
obj_setting["zonal"] = 1.0

# write the output folder path
output_path = "./user_data/output/"

# run the cgodme optimizer to calibrate path flows that can fit the observed target 
# (origin-destination, link volumes, and regional level VMT)
run_optimization(init_od_volume, 
                 layer_mapping_matrix, 
                 path_link_inc, 
                 init_path_flow, 
                 bpr_params, 
                 optimization_setting, 
                 target_data, 
                 obj_setting, 
                 data_source, 
                 output_path)


Imputing the od flow target data to remove unmapped OD pairs...
 - Before removing, the unmapped target od volume is 324540
 - After processing, the target od volume is 300780
Number of Origins: 24
Number of Origin-Destination Pairs: 493
Number of Paths: 1536
Number of Links: 76
Metal device set to: Apple M2


2024-09-26 22:17:39.909279: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2024-09-26 22:17:39.909699: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2024-09-26 22:17:40,148 - Finding Optimal Path Flows...


Epoch 200, Loss: 3.7644693851470947
Epoch 400, Loss: 3.5531697273254395
Epoch 600, Loss: 3.3656375408172607
Epoch 800, Loss: 3.1996874809265137
Epoch 1000, Loss: 3.0532028675079346
Epoch 1200, Loss: 2.924168109893799
Epoch 1400, Loss: 2.8106374740600586
Epoch 1600, Loss: 2.71073579788208
Epoch 1800, Loss: 2.6226587295532227
Epoch 2000, Loss: 2.544708251953125


2024-09-26 22:18:18,143 - Saving loss ...
2024-09-26 22:18:18,146 - Saving the calibrated o flows ...
2024-09-26 22:18:18,148 - Saving the calibrated od flows ...
2024-09-26 22:18:18,150 - Saving the calibrated path flows ...
2024-09-26 22:18:18,155 - Saving the calibrated link flows ...
2024-09-26 22:18:18,165 - RMSE: Passenger Car Count: 1482.2764892578125
2024-09-26 22:18:18,166 - RMSE: Passenger Car VMT: 581.773193359375
2024-09-26 22:18:18,166 - RMSE: OD Flow: 595.7362060546875
2024-09-26 22:18:18,166 - RMSE: O Flow: 3122.239501953125
2024-09-26 22:18:18,166 - Complete!
